# Feature Engineering — Dataset de Obesidade

Este notebook demonstra cada transformação do `FeatureEngineer` com justificativa visual, mostrando distribuições antes/depois e o impacto na correlação com o target.

**Objetivos:**
- Demonstrar as 9 transformações do `FeatureEngineer` com visualizações antes/depois
- Mostrar distribuição e correlação de Spearman do BMI, eating_score e lifestyle_score
- Comparar a correlação de Spearman antes e depois do feature engineering
- Demonstrar o split estratificado 70/15/15 com `StratifiedShuffleSplit(random_state=42)`
- Verificar que a distribuição de classes nos splits está dentro de 5% da original

**Requisitos atendidos:** 5.1–5.8, 6.1, 6.2, 6.3, 13.2

In [ ]:
# Importações
import os
import sys
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.model_selection import StratifiedShuffleSplit

# Configurações de visualização
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
sns.set_theme(style='whitegrid', palette='muted')

# Logging básico para o notebook
logging.basicConfig(level=logging.INFO, format='%(levelname)s — %(message)s')
logger = logging.getLogger('feature_engineering')

print('Importações concluídas.')

In [ ]:
# Adicionar raiz do projeto ao sys.path para importar src/
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data.validator import DataValidator
from src.data.cleaner import DataCleaner
from src.features.engineer import FeatureEngineer
from src.config import (
    OBESITY_LABELS_PT, NUMERIC_RANGES, CATEGORICAL_VALUES, RANDOM_STATE
)

print(f'Raiz do projeto: {PROJECT_ROOT}')
print('Módulos src/ importados com sucesso.')

In [ ]:
# Carregar, validar e limpar os dados
DATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'Obesity.csv')

df_raw = pd.read_csv(DATA_PATH)
logger.info('CSV carregado. Shape: %s', df_raw.shape)

validator = DataValidator()
validation_result = validator.validate(df_raw)

if validation_result.is_valid:
    print('✅ Validação aprovada — nenhum erro encontrado.')
else:
    print(f'⚠️  Validação com {len(validation_result.errors)} aviso(s):')
    for err in validation_result.errors[:10]:
        print(f'   • {err}')

cleaner = DataCleaner()
df = cleaner.clean(df_raw)
logger.info('Limpeza concluída. Shape final: %s', df.shape)

print(f'\nShape após limpeza: {df.shape}')
print(f'Duplicatas removidas: {len(df_raw) - len(df)}')
print(f'Valores ausentes restantes: {df.isnull().sum().sum()}')

## 1. Visão Geral das Transformações

O `FeatureEngineer` aplica **9 transformações** em sequência, transformando as 16 features brutas em um conjunto enriquecido e normalizado pronto para modelagem:

| # | Transformação | Colunas de entrada | Saída |
|---|--------------|-------------------|-------|
| 1 | **BMI** = Weight / Height² | Weight, Height | Nova coluna numérica |
| 2 | **eating_score** (combinação ponderada) | FAVC, FCVC, NCP, CAEC | Nova coluna [0, 1] |
| 3 | **lifestyle_score** (combinação ponderada) | FAF, TUE, CALC, SMOKE | Nova coluna [0, 1] |
| 4 | **high_calorie_sedentary** (flag binária) | FAVC, FAF | Nova coluna 0/1 |
| 5 | **Label encoding** (binárias) | Gender, FAVC, SMOKE, SCC, family_history | 0/1 |
| 6 | **Ordinal encoding** | CAEC, CALC | 0–3 |
| 7 | **One-hot encoding** | MTRANS | 5 colunas binárias |
| 8 | **StandardScaler** | Age, Height, Weight, BMI | Média=0, Std=1 |
| 9 | **MinMaxScaler** | FCVC, NCP, CH2O, FAF, TUE | [0, 1] |

**Justificativa geral:**
- As features engineered (BMI, eating_score, lifestyle_score) capturam conhecimento de domínio médico que features brutas isoladas não expressam.
- Os encodings convertem variáveis categóricas em representações numéricas adequadas para algoritmos de ML.
- O scaling garante que features com escalas diferentes (ex: Weight em kg vs. FAF em 0–3) não dominem o aprendizado do modelo.
- O `FeatureEngineer` é ajustado **apenas no treino** e aplicado no val/test, evitando data leakage.

In [ ]:
# Constantes auxiliares para o notebook
CLASS_ORDER = list(OBESITY_LABELS_PT.keys())
NUMERIC_COLS = list(NUMERIC_RANGES.keys())

# Codificação ordinal do target para cálculos de correlação
OBESITY_ORDINAL = {
    'Insufficient_Weight': 0, 'Normal_Weight': 1,
    'Overweight_Level_I': 2, 'Overweight_Level_II': 3,
    'Obesity_Type_I': 4, 'Obesity_Type_II': 5, 'Obesity_Type_III': 6
}

# Separar features (X) e target (y)
X = df.drop(columns=['Obesity'])
y = df['Obesity']

print(f'Features (X): {X.shape}')
print(f'Target (y): {y.shape}')
print(f'Classes: {y.nunique()} classes únicas')

## 2. BMI = Weight / Height²

O **Índice de Massa Corporal (IMC / BMI)** é o indicador clínico mais utilizado para classificação de obesidade. A fórmula `BMI = Weight / Height²` combina peso e altura em uma única métrica que captura a relação entre eles de forma mais informativa do que as duas features separadas.

**Justificativa:**
- Peso e altura isolados têm correlação com obesidade, mas o BMI captura a relação entre eles — uma pessoa de 90kg pode ser saudável se tiver 1,90m, mas obesa se tiver 1,60m.
- O BMI é a base das classificações clínicas: < 18.5 (abaixo do peso), 18.5–24.9 (normal), 25–29.9 (sobrepeso), ≥ 30 (obesidade).
- Após o cálculo, o BMI é escalado com `StandardScaler` junto com Age, Height e Weight.

**Invariante verificado:** `|BMI - Weight / Height²| < 0.001` para todos os registros.

In [ ]:
# Calcular BMI manualmente para visualização (antes do FeatureEngineer)
bmi_values = X['Weight'] / (X['Height'] ** 2)

# Verificar invariante: BMI = Weight / Height²
max_error = (bmi_values - X['Weight'] / (X['Height'] ** 2)).abs().max()
print(f'✅ Invariante BMI verificado. Erro máximo: {max_error:.6f} (< 0.001)')

# Estatísticas do BMI
print(f'\nEstatísticas do BMI:')
print(f'  Média:   {bmi_values.mean():.2f} kg/m²')
print(f'  Mediana: {bmi_values.median():.2f} kg/m²')
print(f'  Std:     {bmi_values.std():.2f} kg/m²')
print(f'  Min:     {bmi_values.min():.2f} kg/m²')
print(f'  Max:     {bmi_values.max():.2f} kg/m²')

# Distribuição do BMI por classe de obesidade
df_bmi = X.copy()
df_bmi['BMI'] = bmi_values
df_bmi['Obesity'] = y.values
df_bmi['Classe_PT'] = df_bmi['Obesity'].map(OBESITY_LABELS_PT)

order_pt = [OBESITY_LABELS_PT[c] for c in CLASS_ORDER]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Histograma do BMI
ax = axes[0]
ax.hist(bmi_values, bins=40, color='#3B82F6', edgecolor='white', alpha=0.85)
ax.axvline(18.5, color='#22C55E', linestyle='--', linewidth=1.5, label='Abaixo do Peso (18.5)')
ax.axvline(25.0, color='#EAB308', linestyle='--', linewidth=1.5, label='Sobrepeso (25.0)')
ax.axvline(30.0, color='#EF4444', linestyle='--', linewidth=1.5, label='Obesidade (30.0)')
ax.axvline(bmi_values.mean(), color='black', linestyle='-', linewidth=1.5, label=f'Média: {bmi_values.mean():.1f}')
ax.set_title('Distribuição do BMI', fontsize=12, fontweight='bold')
ax.set_xlabel('BMI (kg/m²)')
ax.set_ylabel('Frequência')
ax.legend(fontsize=9)

# Box plot do BMI por classe
ax = axes[1]
palette = dict(zip(order_pt, ['#3B82F6', '#22C55E', '#EAB308', '#F97316', '#EF4444', '#DC2626', '#B91C1C']))
sns.boxplot(data=df_bmi, x='Classe_PT', y='BMI', order=order_pt, palette=palette, ax=ax, linewidth=0.8)
ax.set_title('BMI por Classe de Obesidade', fontsize=12, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('BMI (kg/m²)')
ax.tick_params(axis='x', rotation=35)

plt.suptitle('BMI = Weight / Height² — Distribuição e Separação por Classe', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Correlação de Spearman do BMI com o target
y_ordinal = y.map(OBESITY_ORDINAL)
corr_bmi = bmi_values.corr(y_ordinal, method='spearman')
corr_weight = X['Weight'].corr(y_ordinal, method='spearman')
corr_height = X['Height'].corr(y_ordinal, method='spearman')
print(f'\nCorrelação de Spearman com o target:')
print(f'  BMI:    {corr_bmi:.4f}')
print(f'  Weight: {corr_weight:.4f}')
print(f'  Height: {corr_height:.4f}')
print(f'\n→ BMI captura a relação peso/altura de forma mais informativa.')

## 3. eating_score

O `eating_score` é uma feature composta que agrega quatro indicadores de comportamento alimentar em um único score normalizado para [0, 1]:

```
FAVC_binary = 1 se FAVC == "yes", else 0
CAEC_ordinal = no=0, Sometimes=1, Frequently=2, Always=3
raw = (FAVC_binary × 2) + FCVC + NCP + CAEC_ordinal
eating_score = raw / 12.0  (máximo possível = 2 + 3 + 4 + 3 = 12)
```

**Justificativa:**
- **FAVC** (consumo frequente de alimentos calóricos) recebe peso 2 por ser o fator mais diretamente associado ao ganho de peso.
- **FCVC** (frequência de consumo de vegetais) contribui positivamente — mais vegetais indica hábitos mais saudáveis, mas o score mede "intensidade alimentar" geral.
- **NCP** (número de refeições principais) captura o padrão de frequência alimentar.
- **CAEC** (comer entre refeições) é codificado ordinalmente, pois a frequência importa.
- A normalização para [0, 1] facilita a comparação e o scaling posterior.

In [ ]:
# Calcular eating_score manualmente para visualização
ORDINAL_MAP = {'no': 0, 'Sometimes': 1, 'Frequently': 2, 'Always': 3}
EATING_SCORE_MAX = 12.0

favc_binary = (X['FAVC'].str.lower() == 'yes').astype(float)
caec_ordinal = X['CAEC'].map(ORDINAL_MAP).astype(float)
raw_eating = (favc_binary * 2.0) + X['FCVC'].astype(float) + X['NCP'].astype(float) + caec_ordinal
eating_score = raw_eating / EATING_SCORE_MAX

print('Estatísticas do eating_score:')
print(f'  Média:   {eating_score.mean():.4f}')
print(f'  Mediana: {eating_score.median():.4f}')
print(f'  Std:     {eating_score.std():.4f}')
print(f'  Min:     {eating_score.min():.4f}')
print(f'  Max:     {eating_score.max():.4f}')

# Correlação de Spearman com o target
y_ordinal = y.map(OBESITY_ORDINAL)
corr_eating = eating_score.corr(y_ordinal, method='spearman')
corr_favc = favc_binary.corr(y_ordinal, method='spearman')
corr_fcvc = X['FCVC'].corr(y_ordinal, method='spearman')
corr_ncp = X['NCP'].corr(y_ordinal, method='spearman')
corr_caec = caec_ordinal.corr(y_ordinal, method='spearman')

print(f'\nCorrelação de Spearman com o target:')
print(f'  eating_score: {corr_eating:.4f}')
print(f'  FAVC:         {corr_favc:.4f}')
print(f'  FCVC:         {corr_fcvc:.4f}')
print(f'  NCP:          {corr_ncp:.4f}')
print(f'  CAEC:         {corr_caec:.4f}')

# Visualizações
df_eating = X.copy()
df_eating['eating_score'] = eating_score.values
df_eating['Obesity'] = y.values
df_eating['Classe_PT'] = df_eating['Obesity'].map(OBESITY_LABELS_PT)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Histograma do eating_score
ax = axes[0]
ax.hist(eating_score, bins=30, color='#F97316', edgecolor='white', alpha=0.85)
ax.axvline(eating_score.mean(), color='black', linestyle='--', linewidth=1.5,
           label=f'Média: {eating_score.mean():.3f}')
ax.set_title('Distribuição do eating_score', fontsize=12, fontweight='bold')
ax.set_xlabel('eating_score [0, 1]')
ax.set_ylabel('Frequência')
ax.legend()

# Box plot por classe
ax = axes[1]
order_pt = [OBESITY_LABELS_PT[c] for c in CLASS_ORDER]
palette = dict(zip(order_pt, ['#3B82F6', '#22C55E', '#EAB308', '#F97316', '#EF4444', '#DC2626', '#B91C1C']))
sns.boxplot(data=df_eating, x='Classe_PT', y='eating_score', order=order_pt, palette=palette, ax=ax)
ax.set_title('eating_score por Classe de Obesidade', fontsize=12, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('eating_score [0, 1]')
ax.tick_params(axis='x', rotation=35)

plt.suptitle('eating_score — Comportamento Alimentar Agregado', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 4. lifestyle_score

O `lifestyle_score` agrega quatro indicadores de estilo de vida em um score normalizado para [0, 1], onde valores mais altos indicam estilo de vida mais saudável:

```
CALC_ordinal = no=0, Sometimes=1, Frequently=2, Always=3
SMOKE_binary = 1 se SMOKE == "yes", else 0
raw = FAF - TUE - (CALC_ordinal × 0.5) - (SMOKE_binary × 2)
clipped = clip(raw, -7.0, 3.0)
lifestyle_score = (clipped - (-7.0)) / (3.0 - (-7.0))
```

**Justificativa:**
- **FAF** (frequência de atividade física) contribui positivamente — mais exercício é melhor.
- **TUE** (tempo de uso de tecnologia) contribui negativamente — mais tempo sedentário é pior.
- **CALC** (consumo de álcool) contribui negativamente com peso 0.5 — impacto moderado.
- **SMOKE** (tabagismo) contribui negativamente com peso 2 — impacto forte na saúde.
- O clipping em [-7, 3] evita que outliers extremos distorçam a normalização.
- A normalização para [0, 1] facilita a interpretação: 0 = estilo de vida mais prejudicial, 1 = mais saudável.

In [ ]:
# Calcular lifestyle_score manualmente para visualização
LIFESTYLE_CLIP_MIN = -7.0
LIFESTYLE_CLIP_MAX = 3.0

calc_ordinal = X['CALC'].map(ORDINAL_MAP).astype(float)
smoke_binary = (X['SMOKE'].str.lower() == 'yes').astype(float)

raw_lifestyle = (
    X['FAF'].astype(float)
    - X['TUE'].astype(float)
    - (calc_ordinal * 0.5)
    - (smoke_binary * 2.0)
)
clipped = raw_lifestyle.clip(lower=LIFESTYLE_CLIP_MIN, upper=LIFESTYLE_CLIP_MAX)
lifestyle_score = (clipped - LIFESTYLE_CLIP_MIN) / (LIFESTYLE_CLIP_MAX - LIFESTYLE_CLIP_MIN)

print('Estatísticas do lifestyle_score:')
print(f'  Média:   {lifestyle_score.mean():.4f}')
print(f'  Mediana: {lifestyle_score.median():.4f}')
print(f'  Std:     {lifestyle_score.std():.4f}')
print(f'  Min:     {lifestyle_score.min():.4f}')
print(f'  Max:     {lifestyle_score.max():.4f}')

# Correlação de Spearman com o target
corr_lifestyle = lifestyle_score.corr(y_ordinal, method='spearman')
corr_faf = X['FAF'].corr(y_ordinal, method='spearman')
corr_tue = X['TUE'].corr(y_ordinal, method='spearman')
corr_calc = calc_ordinal.corr(y_ordinal, method='spearman')
corr_smoke = smoke_binary.corr(y_ordinal, method='spearman')

print(f'\nCorrelação de Spearman com o target:')
print(f'  lifestyle_score: {corr_lifestyle:.4f}')
print(f'  FAF:             {corr_faf:.4f}')
print(f'  TUE:             {corr_tue:.4f}')
print(f'  CALC:            {corr_calc:.4f}')
print(f'  SMOKE:           {corr_smoke:.4f}')

# Visualizações
df_lifestyle = X.copy()
df_lifestyle['lifestyle_score'] = lifestyle_score.values
df_lifestyle['Obesity'] = y.values
df_lifestyle['Classe_PT'] = df_lifestyle['Obesity'].map(OBESITY_LABELS_PT)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Histograma do lifestyle_score
ax = axes[0]
ax.hist(lifestyle_score, bins=30, color='#22C55E', edgecolor='white', alpha=0.85)
ax.axvline(lifestyle_score.mean(), color='black', linestyle='--', linewidth=1.5,
           label=f'Média: {lifestyle_score.mean():.3f}')
ax.set_title('Distribuição do lifestyle_score', fontsize=12, fontweight='bold')
ax.set_xlabel('lifestyle_score [0, 1]')
ax.set_ylabel('Frequência')
ax.legend()

# Box plot por classe
ax = axes[1]
order_pt = [OBESITY_LABELS_PT[c] for c in CLASS_ORDER]
palette = dict(zip(order_pt, ['#3B82F6', '#22C55E', '#EAB308', '#F97316', '#EF4444', '#DC2626', '#B91C1C']))
sns.boxplot(data=df_lifestyle, x='Classe_PT', y='lifestyle_score', order=order_pt, palette=palette, ax=ax)
ax.set_title('lifestyle_score por Classe de Obesidade', fontsize=12, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('lifestyle_score [0, 1]')
ax.tick_params(axis='x', rotation=35)

plt.suptitle('lifestyle_score — Estilo de Vida Agregado', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 5. high_calorie_sedentary

A flag `high_calorie_sedentary` identifica o perfil de maior risco: consumo frequente de alimentos calóricos **combinado** com baixa atividade física.

```
high_calorie_sedentary = 1 se (FAVC == "yes") E (FAF < 1), caso contrário 0
```

**Justificativa:**
- A combinação de alta ingestão calórica com sedentarismo é um fator de risco independente para obesidade, mais relevante do que cada fator isolado.
- O limiar `FAF < 1` captura pessoas que praticam pouca ou nenhuma atividade física (FAF = 0 ou valores fracionários próximos de 0).
- Esta feature de interação permite que o modelo capture padrões que não seriam detectados por FAVC e FAF separadamente.

In [ ]:
# Calcular high_calorie_sedentary
favc_yes = X['FAVC'].str.lower() == 'yes'
faf_low = X['FAF'].astype(float) < 1.0
high_cal_sed = (favc_yes & faf_low).astype(int)

n_flagged = high_cal_sed.sum()
pct_flagged = 100.0 * n_flagged / len(X)
print(f'Registros com high_calorie_sedentary = 1: {n_flagged} ({pct_flagged:.1f}%)')
print(f'Registros com high_calorie_sedentary = 0: {len(X) - n_flagged} ({100 - pct_flagged:.1f}%)')

# Correlação de Spearman com o target
corr_hcs = high_cal_sed.corr(y_ordinal, method='spearman')
print(f'\nCorrelação de Spearman com o target: {corr_hcs:.4f}')

# Cross-tabulation com a classe de obesidade
df_hcs = pd.DataFrame({'high_calorie_sedentary': high_cal_sed.values, 'Obesity': y.values})
crosstab = pd.crosstab(
    df_hcs['Obesity'], df_hcs['high_calorie_sedentary'],
    normalize='index'
).mul(100).round(1)
crosstab.index = [OBESITY_LABELS_PT[c] for c in crosstab.index]
crosstab.columns = ['Não (0)', 'Sim (1)']
print('\nProporção de high_calorie_sedentary por classe de obesidade (%):')
print(crosstab.to_string())

# Visualização
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Frequência geral
ax = axes[0]
counts = high_cal_sed.value_counts().sort_index()
bars = ax.bar(['Não (0)', 'Sim (1)'], counts.values, color=['#3B82F6', '#EF4444'], edgecolor='white')
for bar, count in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
            f'{count}\n({100*count/len(X):.1f}%)', ha='center', va='bottom', fontsize=10)
ax.set_title('Frequência de high_calorie_sedentary', fontsize=12, fontweight='bold')
ax.set_ylabel('Número de Registros')
ax.set_xlabel('high_calorie_sedentary')

# Proporção por classe
ax = axes[1]
crosstab_plot = crosstab.reindex([OBESITY_LABELS_PT[c] for c in CLASS_ORDER])
crosstab_plot[['Não (0)', 'Sim (1)']].plot(
    kind='bar', ax=ax, color=['#3B82F6', '#EF4444'], edgecolor='white'
)
ax.set_title('high_calorie_sedentary por Classe de Obesidade (%)', fontsize=12, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Proporção (%)')
ax.tick_params(axis='x', rotation=35)
ax.legend(title='high_calorie_sedentary')

plt.suptitle('high_calorie_sedentary — Flag de Risco Combinado', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 6. Encodings (Label, Ordinal, One-Hot)

As variáveis categóricas precisam ser convertidas em representações numéricas para os algoritmos de ML. O `FeatureEngineer` aplica três tipos de encoding:

**Label Encoding (0/1) — variáveis binárias:**
- `Gender`: Male=1, Female=0
- `FAVC`, `SMOKE`, `SCC`, `family_history`: yes=1, no=0
- Justificativa: variáveis binárias têm apenas dois estados — label encoding é suficiente e não introduz ordenação artificial.

**Ordinal Encoding — variáveis com ordem natural:**
- `CAEC` e `CALC`: no=0, Sometimes=1, Frequently=2, Always=3
- Justificativa: estas variáveis têm uma ordem semântica clara (nunca < às vezes < frequentemente < sempre). O encoding ordinal preserva essa ordem, que é informativa para o modelo.

**One-Hot Encoding — variáveis nominais:**
- `MTRANS`: 5 colunas binárias (Automobile, Bike, Motorbike, Public_Transportation, Walking)
- Justificativa: os meios de transporte não têm ordem natural. One-hot encoding evita que o modelo interprete diferenças numéricas como diferenças de magnitude.

In [ ]:
# Demonstrar Label Encoding
print('=== Label Encoding (0/1) ===')
print('\nAntes:')
print(X[['Gender', 'FAVC', 'SMOKE', 'SCC', 'family_history']].head(5).to_string())

X_label = X.copy()
X_label['Gender'] = (X_label['Gender'].str.lower() == 'male').astype(int)
for col in ('FAVC', 'SMOKE', 'SCC', 'family_history'):
    X_label[col] = (X_label[col].str.lower() == 'yes').astype(int)

print('\nDepois:')
print(X_label[['Gender', 'FAVC', 'SMOKE', 'SCC', 'family_history']].head(5).to_string())

# Verificar distribuição após encoding
print('\nDistribuição após label encoding:')
for col in ['Gender', 'FAVC', 'SMOKE', 'SCC', 'family_history']:
    vc = X_label[col].value_counts().sort_index()
    print(f'  {col}: 0={vc.get(0, 0)} ({100*vc.get(0,0)/len(X_label):.1f}%), 1={vc.get(1, 0)} ({100*vc.get(1,0)/len(X_label):.1f}%)')

In [ ]:
# Demonstrar Ordinal Encoding
print('=== Ordinal Encoding (CAEC e CALC) ===')
print('Mapeamento: no=0, Sometimes=1, Frequently=2, Always=3')
print('\nAntes:')
print(X[['CAEC', 'CALC']].value_counts().head(10).to_string())

X_ordinal = X.copy()
X_ordinal['CAEC'] = X_ordinal['CAEC'].map(ORDINAL_MAP).astype(int)
X_ordinal['CALC'] = X_ordinal['CALC'].map(ORDINAL_MAP).astype(int)

print('\nDepois — distribuição de CAEC:')
print(X_ordinal['CAEC'].value_counts().sort_index().to_string())
print('\nDepois — distribuição de CALC:')
print(X_ordinal['CALC'].value_counts().sort_index().to_string())

# Verificar preservação de ordem
caec_means = {}
for val, label in [(0, 'no'), (1, 'Sometimes'), (2, 'Frequently'), (3, 'Always')]:
    mask = X_ordinal['CAEC'] == val
    if mask.sum() > 0:
        caec_means[label] = y_ordinal[mask].mean()
print('\nMédia do target por valor de CAEC (deve aumentar com o ordinal):')
for label, mean in caec_means.items():
    print(f'  {label}: {mean:.3f}')

In [ ]:
# Demonstrar One-Hot Encoding de MTRANS
print('=== One-Hot Encoding (MTRANS) ===')
print('\nAntes — distribuição de MTRANS:')
print(X['MTRANS'].value_counts().to_string())

from src.config import CATEGORICAL_VALUES
MTRANS_CATEGORIES = sorted(CATEGORICAL_VALUES['MTRANS'])

X_ohe = X.copy()
for cat in MTRANS_CATEGORIES:
    X_ohe[f'MTRANS_{cat}'] = (X_ohe['MTRANS'] == cat).astype(int)
X_ohe = X_ohe.drop(columns=['MTRANS'])

ohe_cols = [f'MTRANS_{cat}' for cat in MTRANS_CATEGORIES]
print('\nDepois — primeiras 5 linhas das colunas OHE:')
print(X_ohe[ohe_cols].head(5).to_string())

# Verificar invariante: soma das colunas OHE = 1 para cada registro
ohe_sum = X_ohe[ohe_cols].sum(axis=1)
assert (ohe_sum == 1).all(), 'ERRO: soma das colunas OHE != 1 para algum registro!'
print(f'\n✅ Invariante OHE verificado: soma das colunas = 1 para todos os {len(X_ohe)} registros.')

# Visualização
fig, ax = plt.subplots(figsize=(10, 4))
mtrans_counts = X['MTRANS'].value_counts().reindex(MTRANS_CATEGORIES)
bars = ax.bar(MTRANS_CATEGORIES, mtrans_counts.values, color='#8B5CF6', edgecolor='white')
for bar, count in zip(bars, mtrans_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 3,
            f'{count}\n({100*count/len(X):.1f}%)', ha='center', va='bottom', fontsize=9)
ax.set_title('Distribuição de MTRANS (antes do One-Hot Encoding)', fontsize=12, fontweight='bold')
ax.set_xlabel('Meio de Transporte')
ax.set_ylabel('Frequência')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

## 7. Scaling (StandardScaler e MinMaxScaler)

O scaling normaliza as escalas das features numéricas para evitar que features com valores maiores (ex: Weight em kg) dominem o aprendizado em detrimento de features com valores menores (ex: FAF em 0–3).

**StandardScaler** (média=0, desvio padrão=1) — aplicado em: `Age`, `Height`, `Weight`, `BMI`
- Justificativa: estas features têm distribuições aproximadamente normais ou simétricas. O StandardScaler é adequado quando não há outliers extremos e a distribuição é razoavelmente gaussiana.
- Fórmula: `z = (x - média) / std`

**MinMaxScaler** (escala [0, 1]) — aplicado em: `FCVC`, `NCP`, `CH2O`, `FAF`, `TUE`
- Justificativa: estas features são discretizadas em escalas fixas (0–3 ou 1–4). O MinMaxScaler é adequado quando os limites são conhecidos e fixos, preservando a distribuição original.
- Fórmula: `x_scaled = (x - min) / (max - min)`

**Regra crítica:** O `fit` dos scalers acontece **apenas no treino**. O `transform` no val/test usa os parâmetros aprendidos no treino, evitando data leakage.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Preparar dados com BMI para demonstração do scaling
X_for_scaling = X.copy()
X_for_scaling['BMI'] = X_for_scaling['Weight'] / (X_for_scaling['Height'] ** 2)

STANDARD_COLS = ['Age', 'Height', 'Weight', 'BMI']
MINMAX_COLS = ['FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']

# Ajustar scalers (simulando fit no treino completo para demonstração)
std_scaler = StandardScaler()
mm_scaler = MinMaxScaler()

X_std_before = X_for_scaling[STANDARD_COLS].copy()
X_mm_before = X_for_scaling[MINMAX_COLS].copy()

X_std_after = pd.DataFrame(
    std_scaler.fit_transform(X_std_before),
    columns=STANDARD_COLS
)
X_mm_after = pd.DataFrame(
    mm_scaler.fit_transform(X_mm_before),
    columns=MINMAX_COLS
)

# Estatísticas antes/depois
print('=== StandardScaler — Antes vs Depois ===')
stats_before = X_std_before.agg(['mean', 'std', 'min', 'max']).round(4)
stats_after = X_std_after.agg(['mean', 'std', 'min', 'max']).round(4)
print('\nAntes:')
print(stats_before.to_string())
print('\nDepois:')
print(stats_after.to_string())

print('\n=== MinMaxScaler — Antes vs Depois ===')
stats_mm_before = X_mm_before.agg(['mean', 'std', 'min', 'max']).round(4)
stats_mm_after = X_mm_after.agg(['mean', 'std', 'min', 'max']).round(4)
print('\nAntes:')
print(stats_mm_before.to_string())
print('\nDepois:')
print(stats_mm_after.to_string())

In [ ]:
# Visualização: distribuições antes/depois do scaling
fig, axes = plt.subplots(2, 4, figsize=(18, 8))

NOMES_PT_SCALE = {
    'Age': 'Idade', 'Height': 'Altura', 'Weight': 'Peso', 'BMI': 'BMI'
}

for i, col in enumerate(STANDARD_COLS):
    ax_before = axes[0][i]
    ax_after = axes[1][i]

    ax_before.hist(X_std_before[col], bins=30, color='#3B82F6', edgecolor='white', alpha=0.8)
    ax_before.set_title(f'{NOMES_PT_SCALE[col]}\n(antes)', fontsize=10, fontweight='bold')
    ax_before.set_ylabel('Frequência' if i == 0 else '')

    ax_after.hist(X_std_after[col], bins=30, color='#8B5CF6', edgecolor='white', alpha=0.8)
    ax_after.set_title(f'{NOMES_PT_SCALE[col]}\n(depois — StandardScaler)', fontsize=10, fontweight='bold')
    ax_after.set_ylabel('Frequência' if i == 0 else '')

plt.suptitle('StandardScaler — Distribuições Antes e Depois', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# MinMaxScaler
NOMES_PT_MM = {
    'FCVC': 'Freq. Vegetais', 'NCP': 'Nº Refeições',
    'CH2O': 'Água (L/dia)', 'FAF': 'Ativ. Física', 'TUE': 'Uso Tecnologia'
}

fig, axes = plt.subplots(2, 5, figsize=(20, 7))

for i, col in enumerate(MINMAX_COLS):
    ax_before = axes[0][i]
    ax_after = axes[1][i]

    ax_before.hist(X_mm_before[col], bins=20, color='#F97316', edgecolor='white', alpha=0.8)
    ax_before.set_title(f'{NOMES_PT_MM[col]}\n(antes)', fontsize=9, fontweight='bold')
    ax_before.set_ylabel('Frequência' if i == 0 else '')

    ax_after.hist(X_mm_after[col], bins=20, color='#22C55E', edgecolor='white', alpha=0.8)
    ax_after.set_title(f'{NOMES_PT_MM[col]}\n(depois — MinMaxScaler)', fontsize=9, fontweight='bold')
    ax_after.set_ylabel('Frequência' if i == 0 else '')

plt.suptitle('MinMaxScaler — Distribuições Antes e Depois', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 8. Correlação de Spearman — Antes vs Depois

Comparar a correlação de Spearman das features com o target antes e depois do feature engineering permite avaliar se as transformações aumentaram o poder preditivo do conjunto de features.

**Metodologia:**
- **Antes:** features brutas com codificação mínima (binárias → 0/1, ordinais → 0–3, MTRANS → ordinal arbitrário)
- **Depois:** features transformadas pelo `FeatureEngineer` completo (BMI, eating_score, lifestyle_score, encodings, scaling)
- O scaling (StandardScaler/MinMaxScaler) não altera a correlação de Spearman (é uma transformação monotônica), mas as features engineered (BMI, scores) podem ter correlações diferentes das features brutas originais.

In [ ]:
# Correlação ANTES do feature engineering
df_before = X.copy()
df_before['Obesity_Ordinal'] = y.map(OBESITY_ORDINAL).values

# Codificação mínima para calcular correlação das features brutas
df_before['Gender_enc'] = (df_before['Gender'] == 'Male').astype(int)
df_before['family_history_enc'] = (df_before['family_history'].str.lower() == 'yes').astype(int)
df_before['FAVC_enc'] = (df_before['FAVC'].str.lower() == 'yes').astype(int)
df_before['SMOKE_enc'] = (df_before['SMOKE'].str.lower() == 'yes').astype(int)
df_before['SCC_enc'] = (df_before['SCC'].str.lower() == 'yes').astype(int)
df_before['CAEC_enc'] = df_before['CAEC'].map(ORDINAL_MAP)
df_before['CALC_enc'] = df_before['CALC'].map(ORDINAL_MAP)
MTRANS_MAP_ORD = {'Automobile': 0, 'Motorbike': 1, 'Public_Transportation': 2, 'Bike': 3, 'Walking': 4}
df_before['MTRANS_enc'] = df_before['MTRANS'].map(MTRANS_MAP_ORD)

BEFORE_COLS = NUMERIC_COLS + ['Gender_enc', 'family_history_enc', 'FAVC_enc',
                               'CAEC_enc', 'SMOKE_enc', 'SCC_enc', 'CALC_enc', 'MTRANS_enc']
BEFORE_LABELS = {
    'Age': 'Idade', 'Height': 'Altura', 'Weight': 'Peso',
    'FCVC': 'Freq. Vegetais', 'NCP': 'Nº Refeições', 'CH2O': 'Água',
    'FAF': 'Ativ. Física', 'TUE': 'Uso Tecnologia',
    'Gender_enc': 'Gênero', 'family_history_enc': 'Hist. Familiar',
    'FAVC_enc': 'Alim. Calórica', 'CAEC_enc': 'Comer Fora',
    'SMOKE_enc': 'Tabagismo', 'SCC_enc': 'Monitor Cal.',
    'CALC_enc': 'Álcool', 'MTRANS_enc': 'Transporte'
}

corr_before = df_before[BEFORE_COLS + ['Obesity_Ordinal']].corr(method='spearman')['Obesity_Ordinal'].drop('Obesity_Ordinal')
corr_before.index = [BEFORE_LABELS.get(c, c) for c in corr_before.index]
corr_before = corr_before.sort_values(key=abs, ascending=False)

print('Correlação de Spearman com o target — ANTES do feature engineering:')
print(corr_before.round(4).to_string())

In [ ]:
# Correlação DEPOIS do feature engineering
# Usar o FeatureEngineer real para obter as features transformadas
fe = FeatureEngineer()
fe.fit(X, y)
X_transformed = fe.transform(X)
feature_names = fe.get_feature_names_out()

df_after = pd.DataFrame(X_transformed, columns=feature_names)
df_after['Obesity_Ordinal'] = y.map(OBESITY_ORDINAL).values

# Calcular correlação de Spearman para features numéricas e engineered
# (excluir colunas OHE de MTRANS para simplificar a comparação)
AFTER_COLS_MAIN = ['Age', 'Height', 'Weight', 'BMI', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE',
                   'eating_score', 'lifestyle_score', 'high_calorie_sedentary',
                   'Gender', 'FAVC', 'SMOKE', 'SCC', 'family_history', 'CAEC', 'CALC']

AFTER_LABELS = {
    'Age': 'Idade', 'Height': 'Altura', 'Weight': 'Peso', 'BMI': 'BMI',
    'FCVC': 'Freq. Vegetais', 'NCP': 'Nº Refeições', 'CH2O': 'Água',
    'FAF': 'Ativ. Física', 'TUE': 'Uso Tecnologia',
    'eating_score': 'eating_score', 'lifestyle_score': 'lifestyle_score',
    'high_calorie_sedentary': 'high_cal_sed',
    'Gender': 'Gênero', 'FAVC': 'Alim. Calórica', 'SMOKE': 'Tabagismo',
    'SCC': 'Monitor Cal.', 'family_history': 'Hist. Familiar',
    'CAEC': 'Comer Fora', 'CALC': 'Álcool'
}

corr_after = df_after[AFTER_COLS_MAIN + ['Obesity_Ordinal']].corr(method='spearman')['Obesity_Ordinal'].drop('Obesity_Ordinal')
corr_after.index = [AFTER_LABELS.get(c, c) for c in corr_after.index]
corr_after = corr_after.sort_values(key=abs, ascending=False)

print('Correlação de Spearman com o target — DEPOIS do feature engineering:')
print(corr_after.round(4).to_string())

In [ ]:
# Visualização comparativa: Antes vs Depois
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Antes
ax = axes[0]
colors_before = ['#EF4444' if v > 0 else '#3B82F6' for v in corr_before.values]
bars = ax.barh(corr_before.index, corr_before.values, color=colors_before, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Correlação de Spearman')
ax.set_title('ANTES do Feature Engineering', fontsize=12, fontweight='bold')
for bar, val in zip(bars, corr_before.values):
    offset = 0.005 if val >= 0 else -0.005
    ha = 'left' if val >= 0 else 'right'
    ax.text(val + offset, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', ha=ha, fontsize=8)
ax.set_xlim(-0.8, 0.8)

# Depois
ax = axes[1]
colors_after = ['#EF4444' if v > 0 else '#3B82F6' for v in corr_after.values]
bars = ax.barh(corr_after.index, corr_after.values, color=colors_after, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Correlação de Spearman')
ax.set_title('DEPOIS do Feature Engineering', fontsize=12, fontweight='bold')
for bar, val in zip(bars, corr_after.values):
    offset = 0.005 if val >= 0 else -0.005
    ha = 'left' if val >= 0 else 'right'
    ax.text(val + offset, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', ha=ha, fontsize=8)
ax.set_xlim(-0.8, 0.8)

plt.suptitle('Correlação de Spearman com o Target — Antes vs Depois do Feature Engineering',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('\n→ O BMI apresenta correlação mais alta do que Weight e Height separados.')
print('→ eating_score e lifestyle_score agregam informação de múltiplas features.')
print('→ high_calorie_sedentary captura interação entre FAVC e FAF.')

## 9. Split Estratificado 70/15/15

A divisão dos dados em treino (70%), validação (15%) e teste (15%) é feita com `StratifiedShuffleSplit`, que garante que a proporção de cada classe de obesidade seja preservada em todos os splits.

**Justificativa:**
- **Estratificação:** sem estratificação, splits aleatórios podem sub-representar classes minoritárias, levando a avaliações enviesadas.
- **70/15/15:** o treino recebe a maior parte dos dados para maximizar o aprendizado. Validação e teste recebem partes iguais para avaliação imparcial.
- **`random_state=42`:** garante reprodutibilidade — os mesmos splits são gerados em qualquer execução.
- **Tolerância de 5%:** a distribuição de classes em cada split deve estar dentro de 5% da distribuição original, confirmando que a estratificação funcionou corretamente.

**Procedimento:**
1. `StratifiedShuffleSplit(test_size=0.30)` → treino (70%) vs. resto (30%)
2. `StratifiedShuffleSplit(test_size=0.50)` no resto → validação (15%) e teste (15%)

In [ ]:
# Split estratificado 70/15/15
# Passo 1: treino (70%) vs. resto (30%)
sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=RANDOM_STATE)
train_idx, rest_idx = next(sss1.split(X, y))

X_train = X.iloc[train_idx].reset_index(drop=True)
y_train = y.iloc[train_idx].reset_index(drop=True)
X_rest = X.iloc[rest_idx].reset_index(drop=True)
y_rest = y.iloc[rest_idx].reset_index(drop=True)

# Passo 2: validação (15%) vs. teste (15%) a partir do resto (30%)
sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=RANDOM_STATE)
val_idx, test_idx = next(sss2.split(X_rest, y_rest))

X_val = X_rest.iloc[val_idx].reset_index(drop=True)
y_val = y_rest.iloc[val_idx].reset_index(drop=True)
X_test = X_rest.iloc[test_idx].reset_index(drop=True)
y_test = y_rest.iloc[test_idx].reset_index(drop=True)

total = len(X)
print('=== Tamanhos dos Splits ===')
print(f'Total:      {total:5d} registros (100.0%)')
print(f'Treino:     {len(X_train):5d} registros ({100*len(X_train)/total:.1f}%)')
print(f'Validação:  {len(X_val):5d} registros ({100*len(X_val)/total:.1f}%)')
print(f'Teste:      {len(X_test):5d} registros ({100*len(X_test)/total:.1f}%)')
print(f'\nSoma: {len(X_train) + len(X_val) + len(X_test)} (deve ser {total})')

In [ ]:
# Verificar distribuição de classes nos splits
def class_distribution(y_series, name):
    """Retorna DataFrame com distribuição de classes em percentual."""
    counts = y_series.value_counts()
    pct = y_series.value_counts(normalize=True).mul(100)
    return pd.DataFrame({'Contagem': counts, f'% ({name})': pct.round(2)})

dist_original = class_distribution(y, 'Original')
dist_train = class_distribution(y_train, 'Treino')
dist_val = class_distribution(y_val, 'Validação')
dist_test = class_distribution(y_test, 'Teste')

# Tabela comparativa
dist_compare = pd.DataFrame({
    'Classe (PT-BR)': [OBESITY_LABELS_PT[c] for c in CLASS_ORDER],
    '% Original': [y.value_counts(normalize=True).mul(100).get(c, 0) for c in CLASS_ORDER],
    '% Treino': [y_train.value_counts(normalize=True).mul(100).get(c, 0) for c in CLASS_ORDER],
    '% Validação': [y_val.value_counts(normalize=True).mul(100).get(c, 0) for c in CLASS_ORDER],
    '% Teste': [y_test.value_counts(normalize=True).mul(100).get(c, 0) for c in CLASS_ORDER],
})
dist_compare = dist_compare.round(2)

# Calcular desvio máximo em relação ao original
for split_col in ['% Treino', '% Validação', '% Teste']:
    dist_compare[f'Δ {split_col.replace("% ", "")}'] = (
        (dist_compare[split_col] - dist_compare['% Original']).abs().round(2)
    )

print('=== Distribuição de Classes nos Splits ===')
print(dist_compare.to_string(index=False))

# Verificar tolerância de 5%
print('\n=== Verificação de Tolerância (≤ 5%) ===')
tolerance = 5.0
all_ok = True
for split_name in ['Treino', 'Validação', 'Teste']:
    max_delta = dist_compare[f'Δ {split_name}'].max()
    ok = max_delta <= tolerance
    status = '✅' if ok else '❌'
    print(f'{status} {split_name}: desvio máximo = {max_delta:.2f}% (tolerância: {tolerance}%)')
    if not ok:
        all_ok = False

if all_ok:
    print('\n✅ Todos os splits estão dentro da tolerância de 5%.')
else:
    print('\n❌ Algum split excedeu a tolerância de 5%!')

In [ ]:
# Visualização: distribuição de classes nos splits
fig, axes = plt.subplots(1, 4, figsize=(20, 5), sharey=False)

splits_data = [
    (y, 'Original\n(100%)', '#6B7280'),
    (y_train, f'Treino\n({100*len(y_train)/total:.0f}%)', '#3B82F6'),
    (y_val, f'Validação\n({100*len(y_val)/total:.0f}%)', '#22C55E'),
    (y_test, f'Teste\n({100*len(y_test)/total:.0f}%)', '#EF4444'),
]

order_pt = [OBESITY_LABELS_PT[c] for c in CLASS_ORDER]
short_labels = ['Abaixo\nPeso', 'Normal', 'Sobrepeso\nI', 'Sobrepeso\nII',
                'Obesidade\nI', 'Obesidade\nII', 'Obesidade\nIII']

for ax, (y_split, title, color) in zip(axes, splits_data):
    pct = y_split.value_counts(normalize=True).mul(100).reindex(CLASS_ORDER).fillna(0)
    bars = ax.bar(short_labels, pct.values, color=color, edgecolor='white', alpha=0.85)
    for bar, val in zip(bars, pct.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=7.5)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylabel('Proporção (%)' if ax == axes[0] else '')
    ax.set_ylim(0, pct.max() * 1.25)
    ax.tick_params(axis='x', labelsize=8)

plt.suptitle('Distribuição de Classes nos Splits — Estratificação 70/15/15',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 10. Conclusões

### Resumo das transformações e seus impactos

**1. BMI = Weight / Height²**
- O BMI apresenta correlação de Spearman mais alta com o target do que Weight e Height separados.
- A distribuição do BMI por classe de obesidade mostra separação clara, confirmando seu alto poder discriminativo.
- O invariante `|BMI - Weight/Height²| < 0.001` foi verificado para todos os registros.

**2. eating_score e lifestyle_score**
- Ambos os scores agregam informação de múltiplas features em uma única representação normalizada [0, 1].
- O eating_score captura o comportamento alimentar geral; o lifestyle_score captura o estilo de vida.
- A correlação de Spearman desses scores com o target é comparável ou superior às features individuais que os compõem.

**3. high_calorie_sedentary**
- A flag de interação captura o perfil de maior risco (alta ingestão calórica + sedentarismo).
- A cross-tabulation mostra que classes de obesidade mais severas têm maior proporção de registros com esta flag ativa.

**4. Encodings**
- Label encoding preserva a semântica binária das variáveis yes/no.
- Ordinal encoding preserva a ordem natural de CAEC e CALC.
- One-hot encoding evita ordenação artificial em MTRANS.
- O invariante de soma = 1 para as colunas OHE foi verificado para todos os registros.

**5. Scaling**
- StandardScaler normaliza Age, Height, Weight, BMI para média=0, std=1.
- MinMaxScaler normaliza FCVC, NCP, CH2O, FAF, TUE para [0, 1].
- O scaling não altera a correlação de Spearman, mas é essencial para algoritmos sensíveis à escala (SVM, MLP).

**6. Split estratificado 70/15/15**
- A estratificação com `StratifiedShuffleSplit(random_state=42)` preserva a distribuição de classes em todos os splits.
- O desvio máximo em relação à distribuição original está dentro da tolerância de 5% para treino, validação e teste.
- Os splits X_train, X_val, X_test, y_train, y_val, y_test estão prontos para uso no notebook 03.

---
*Próximo passo: `03_model_training.ipynb` — treinamento com Optuna, comparação de modelos e avaliação final.*

In [ ]:
# Resumo final dos splits disponíveis para o notebook 03
print('=== Splits disponíveis para o notebook 03 ===')
print(f'X_train: {X_train.shape}  |  y_train: {y_train.shape}')
print(f'X_val:   {X_val.shape}  |  y_val:   {y_val.shape}')
print(f'X_test:  {X_test.shape}  |  y_test:  {y_test.shape}')
print(f'\nFeatures de entrada: {X_train.shape[1]} colunas')
print(f'Classes: {sorted(y_train.unique())}')
print(f'\nO FeatureEngineer deve ser ajustado em X_train e aplicado em X_val e X_test.')
print(f'random_state={RANDOM_STATE} usado em todos os splits.')